[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/diailectics/blob/main/interface.ipynb)

# Persona chat — Runaway frame annotation

Paste an article below. Claude first reads it against the schema in
[runaway_package_annotator_instructions.html](runaway_package_annotator_instructions.html)
and picks out the passages actually worth annotating (most of an article isn't about
the Runaway frame at all) — this replaces manually marking yes/no yourself. Every
persona from [persona_test_prompts.md](persona_test_prompts.md) you select then
comments on each passage, **run statelessly at first** (matches the
`[STATELESS MODE ONLY: ...]` branch of each prompt), in ~60 words: a one-sentence
**summary** understandable on its own (used as the collapsed preview), a short
argument, an optional quote, its own yes/no **stance**, and a one-line closing note.

The article and its comments sit **side by side** - the highlighted article on the
left, a column of per-passage comments on the right - not one below the other.
Each passage's badge is **yes / no / maybe**: "maybe" means the personas split, and
the badge's motivation names who landed where (e.g. "Split - yes: Sycophant, Hegel;
no: Devil's Advocate"). The article highlight color reflects the same thing - green
for a unanimous yes, red for a unanimous no, amber for a split - so you can scan the
whole article for where the personas actually agree or disagree at a glance.

Expand a persona to read its full response, then **write a reply in the box
underneath it** — it's sent back to that same persona as a follow-up turn (with the
original passage, schema, and its own prior response still in context). The
response, stance badge, and summary all update, but nothing is silently
overwritten: **every reply keeps the version it replaced**, so expanding a persona
after several rounds shows the whole thread - its first response, your reply, its
revised response, your next reply, and so on.

A passage worth discussing doesn't have to come from the automatic selection -
below the article there's an "Add a custom passage" form (your own text, your own
yes/no, your own choice of personas). It lands in the **same** comments column
next to the article, alongside the passages Claude picked out itself - there's one
combined section, not two separate ones.

When you're done, the **"Save your work"** button exports everything annotated so
far (the article run and any custom passages) to a timestamped JSON file: each
passage's text, its provisional annotation, and every persona's complete response
history plus the replies that produced each update - the full record, not just
the final state.

Almost everything above - the widgets, the reply wiring, the save logic - lives in
[tools/notebook_ui.py](tools/notebook_ui.py) as `build_interface()`; this notebook
just calls it. See that module if you want to read or extend how the interface is
built.

**Cost note:** the initial run makes `1 + (passages found × personas selected)` API
calls, run concurrently — e.g. 4 passages × 4 personas = 17 calls. Each reply you
send afterward is one more call. Tune "Max passages" and the persona checkboxes to
control cost.

**Before running:** set `ANTHROPIC_API_KEY` in your shell environment, add a `.env`
file in the project root, or drop the key in `api_key.json`'s `anthropic_api_key`
field (`tools/claude_chat.py` checks all three, in that order, and never prints
the key).

**Note on Adorno:** in `persona_test_prompts.md`, Adorno compares a *set* of outlier
cases against the schema, not a single annotation. Here it's scoped down to "does
this one passage, given its annotation, expose a tension in the schema" — see the
comment above `TASK_LINES` in `tools/claude_chat.py`.

## Running this on Google Colab

Click the badge above, or open
`https://colab.research.google.com/github/kasparvonbeelen/diailectics/blob/main/interface.ipynb`
directly. Then just run the cells top to bottom:

1. **"Colab setup"** clones this repo into `/content/diailectics` and installs
   `requirements.txt` (only runs its clone/install steps when it detects it's
   actually on Colab - it's a no-op if you're running locally instead).
2. **"Authentication"** looks for credentials in this order: an `ANTHROPIC_API_KEY`
   already in the environment, a Colab secret named `ANTHROPIC_API_KEY` (key icon
   in the left sidebar - the recommended way, since it's never stored in the
   notebook or repo), an uploaded `api_key.json` (drop it in the Colab file
   browser - either at `/content/api_key.json` or directly into `diailectics/`,
   either location is picked up automatically), or - as a last resort on Colab
   only - a prompt to paste the key directly (kept in memory for the session only).
3. Run the remaining cells as normal.


### Colab setup (no-op if not on Colab)

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/kasparvonbeelen/diailectics.git"
REPO_DIR = "/content/diailectics"

if IN_COLAB:
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
        check=True,
    )

    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Running on Colab. Cloned/updated {REPO_DIR}, installed requirements, cwd set there.")
else:
    print("Not running on Colab - using the local checkout as-is (pip install -r requirements.txt yourself if needed).")


### Authentication

In [ ]:
from tools.notebook_ui import ensure_api_key

print(ensure_api_key())


In [ ]:
from IPython.display import display, HTML

from tools.claude_chat import ChatBackend
from tools.notebook_ui import build_interface


In [ ]:
# Loads persona_test_prompts.md + runaway_package_annotator_instructions.html once,
# and creates the Anthropic client (see the credential order in the module docstring).
backend = ChatBackend()
print("Personas loaded:", list(backend.personas.keys()))
print("Schema loaded:", len(backend.schema), "characters")
print("Default model:", backend.model)


In [ ]:
ui = build_interface(backend)
display(ui)


## Calling it without the widgets

`tools/notebook_ui.py`'s `build_interface()` is just a thin ipywidgets layer
over `tools/claude_chat.py`'s `ChatBackend` - everything it does is also
available as plain function calls, e.g. for batch/scripted use:

```python
from IPython.display import display, HTML
from tools.claude_chat import render_html, render_conversation_thread

# Whole article - Claude selects the passages itself
result = backend.annotate_article(
    article_text,
    personas=["devils_advocate", "hegel"],  # default: all four
    max_passages=4,
    model="claude-opus-5",                  # optional override
)
display(HTML(result["html"]))  # static hover-popover version, with stance badges

# Every persona output carries its own stance and a conversation you can reply to:
pdata = result["passages"][0]["personas"]["hegel"]
print(pdata["raw_text"])          # includes a <summary> and a <stance>yes|no</stance> tag
updated_raw = pdata["conversation"].reply("I don't think this reading holds up because ...")
display(HTML(render_conversation_thread("hegel", pdata["conversation"])))  # the whole thread so far

# A passage you pick yourself, multiple personas at once (what "Add a custom
# passage" in the interface does) - for text the auto-selector skipped:
passage = backend.run_passage(
    "A sentence the automatic selection step didn't flag as important...",
    personas=["hegel", "adorno"],           # default: all four
    contains_runaway=False,
    justification="Optional - your own reasoning, passed to the personas.",
    model="claude-opus-5",                  # optional override
)
display(HTML(render_html("hegel", passage["personas"]["hegel"]["raw_text"])))

# Single passage, single persona, with a yes/no annotation you already have
result = backend.run(
    text="Le réacteur est hors de contrôle...",
    persona="devils_advocate",
    contains_runaway=True,
    model="claude-sonnet-5",                # optional override
)
display(HTML(result["html"]))
result["conversation"].reply("...")         # this one supports replies too

# Everything the interface itself has produced this session:
for entry in ui.session_passages:           # [{"label", "source", "passage"}, ...]
    print(entry["label"], entry["source"], entry["passage"]["initial_consensus_label"])
```